# Allocation v5

This experiment reads verified tasks and builds a tentative workshop plan. It checks category, workshop status, batch caps and request restrictions before estimating delivery and cost. Only an allowed on-time option can become an automatic recommendation.

The plan uses earliest deadline first. Each recommendation adds its production and expected rework time to a temporary queue. Reviews reserve nothing, and another plan starts from the original queues.

This is a copy of `backend/experimentation_notebooks/03-order-allocation-v5.ipynb`, prepared on 22 October 2026. The source is unchanged. It retains the older processing handoff and policy; the [current backend](../../backend/README.md) documents later changes.

## Reading and running

Sections 1-6 define the rules and show small examples. Section 7 contains controlled checks C1-C14. Section 8 reads the processing handoff and saves the real plan; Section 9 inspects a result.

This notebook makes no model calls. Use the environment in `backend/`. To run the full notebook, first produce the matching handoff with [notebook 02](02-order-processing-from-chat-v4.ipynb). The controlled checks need the definition cells and the Section 5 example cell that defines `check_task`, but no handoff. The public repository does not contain local run files.

This copy reads processing output and writes allocation plans under `experiments/notebooks/output/`. Its code fingerprint reads this copy. Save it before creating a plan after an edit.

Saved cell outputs were removed. Calculations, ranking, checks and decision functions were retained. Earlier fix tags remain in the comments and the table below.

## Earlier fix tags

| Fix | What it does |
|---|---|
| A1, B3 | A deadline review names the closest option (the least-late allowed workshop) and its lateness. A deadline on or before the dispatch date is stated as such, not as negative days. |
| A2 | A question with missing details or doubts goes to `needs_clarification`, not `reference_tasks`. |
| A3 | A hard-rule review names the requested workshop and its failed rule, or lists each rule with the workshops it removed. |
| A4 | Each decision records the order's status and production stage. A COMPLETE order gets a warning. |
| A6, B12 | A shorter trace, and a final table (`summary_rows`) with one row per task or message, its reason and every chat message it covers. Workshop profiles are saved once per plan. |
| A7, B13 | `PLAN_OBJECTIVE` applies one objective to the whole plan, resolved and checked once. `None` keeps each request's preference. |
| A8, B15 | The inspect cell searches every list and prints the outcome and reason before the full record. |
| A9 | The handoff's `intake_review` list must match the saved processing state. |
| B1 | A plan made with `PLAN_OBJECTIVE` is saved to its own file, so the default plan is never overwritten. |
| B2 | A saved plan records `code_hash`, `handoff_state_hash` and `data_hashes`. |
| B4 | An incomplete processing run is reported with its status and verified count. |
| B5 | Non-task messages are listed in the plan and the table. |
| B6, B9, B10, B11 | Every review, clarification and reference row says what is missing, unsupported or unanswered. A duplicate-order review names all the requests. |
| B7 | A recommendation reason quotes the delivery days, days available, cost, ranking and any late workshops removed. |
| B8 | When the requested workshop cannot take the batch, the review also names the best on-time option without the request. Information only: nothing is reserved and the request is not replaced. |
| B14 | Small clean-ups. |
| B16 | Controlled checks run before the real plan. |
| B17 | A top-ranked workshop that the request mentions only as a reference (neither requested nor excluded) is not recommended. The task goes to review so the dispatcher confirms the instruction. Found in processing run 3 (R12). |
| EDD | New in v5. `plan_tasks` plans tasks earliest deadline first instead of in chat order, so a request that can wait cannot take queue time before an urgent one. Ties keep chat order; tasks with no deadline go last; the plan saves `planning_order`. |


## 1. Load the data and settings

- `workshops.csv` and `orders.csv` are only read. Their text is converted once (numbers, sets, dates), and every later rule and formula uses the converted values.
- Capacity is pieces per day, not a balance to subtract from. A queue is days of work already waiting.
- Every request is planned from one snapshot date, `DISPATCH_DATE`.
- Each request's preference selects its ranking objective (Section 4): none or fastest gives `minimize_tardiness`, cheapest gives `minimize_cost`, quality gives `minimize_defects`.
- `PLAN_OBJECTIVE = None` keeps each request's preference. Set it to an objective name (for example `"hybrid"`) to rank every request with that objective instead, as an experiment (A7). That plan is saved to its own file (Section 8).

| Setting | Value | Meaning |
|---|---|---|
| `PROCESSING_DIR` | `output/order-processing-v3` | Where processing v3 and v4 save their runs |
| `OUTPUT_DIR` | `output/order-allocation-v5` | Where this notebook saves its plans |
| `NOTEBOOK_FILE` | `03-order-allocation-v5.ipynb` | This notebook; its `definitions` cells are fingerprinted into each plan (B2) |
| `DISPATCH_DATE` | 2026-04-01 | Snapshot date for every request |
| `DEFAULT_OBJECTIVE` | `minimize_tardiness` | Objective for a request without a preference |
| `PLAN_OBJECTIVE` | `None` | One objective for the whole plan, for experiments (A7) |
| `POLICY_VERSION` | `"v5"` | Label saved with each decision; v5 adds the earliest-deadline-first planning order (Section 6) |

The run cell shows: the counts and settings, one line per workshop, one order row as JSON, and the input and output folders.

In [ ]:
# Section 1 definitions: imports, settings, and the workshop and order data.
# The CSV files are only read. Every later rule and formula uses the converted values below.
import csv
import hashlib
import json
from collections import Counter
from dataclasses import asdict, dataclass
from datetime import date
from pathlib import Path

# Find the project folder. This works from the repository root or from the notebook folder.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "backend/pyproject.toml").is_file() and (p / "backend/data/orders.csv").is_file())
# Input: the processing runs. Processing v3 and v4 both save their runs in this folder.
PROCESSING_DIR = ROOT / "experiments/notebooks/output/order-processing-v3"
# Output: this notebook's own folder, so plans saved by earlier versions are never overwritten.
OUTPUT_DIR = ROOT / "experiments/notebooks/output/order-allocation-v5"
# This notebook's file. Its definitions cells are fingerprinted into every saved plan (B2).
NOTEBOOK_FILE = ROOT / "experiments/notebooks/03-order-allocation-v5.ipynb"
# Every request is planned from this snapshot date.
DISPATCH_DATE = date(2026, 4, 1)
# The objective for a request without a preference.
DEFAULT_OBJECTIVE = "minimize_tardiness"
# A7: None keeps each request's own preference. Set an objective name from Section 4
# (for example "hybrid") to rank every request with that objective instead.
PLAN_OBJECTIVE = None
# The label saved with each decision. v5 adds one planning rule, earliest due date first (EDD,
# Section 6). Every other decision rule is v4's.
POLICY_VERSION = "v5"


# One workshop: only the CSV fields that the rules, formulas and saved records use.
@dataclass
class Workshop:
    workshop_id: str
    name: str
    capacity: int
    lead_days: int
    defect_rate: float
    cost: float
    makes: set[str]
    status: str
    max_batch: int | None
    queue0: float
    notes: str


# Convert the CSV text once: numbers, a set of categories, and None for a blank batch cap.
with (ROOT / "backend/data/workshops.csv").open(newline="", encoding="utf-8") as file:
    WORKSHOPS = {
        row["workshop_id"]: Workshop(
            workshop_id=row["workshop_id"], name=row["name"],
            capacity=int(row["capacity_pieces_per_day"]), lead_days=int(row["pickup_lead_days"]),
            defect_rate=float(row["defect_rate"]), cost=float(row["cost_per_piece"]),
            makes=set(row["makes"].split("+")), status=row["status"],
            max_batch=int(row["max_batch_pieces"]) if row["max_batch_pieces"] else None,
            queue0=float(row["current_queue_days"]), notes=row["notes"])
        for row in csv.DictReader(file)
    }
# Each order keeps every CSV column (A4 uses status and current_stage).
# pieces becomes a number and due_date a date.
with (ROOT / "backend/data/orders.csv").open(newline="", encoding="utf-8") as file:
    ORDERS = {row["order_id"]: {**row, "pieces": int(row["pieces"]),
                               "due_date": date.fromisoformat(row["due_date"])}
              for row in csv.DictReader(file)}
# Each workshop's starting queue, in days. Plans work on a copy; nothing changes this one.
INITIAL_QUEUES = {wid: workshop.queue0 for wid, workshop in WORKSHOPS.items()}

In [ ]:
# Section 1 run: show the data and settings that every rule and formula uses.
print(f"Loaded {len(ORDERS)} orders and {len(WORKSHOPS)} workshops. Dispatch snapshot: {DISPATCH_DATE}")
print(f"Default ranking objective: {DEFAULT_OBJECTIVE}; plan objective setting: {PLAN_OBJECTIVE}")
# One line per workshop. The queue shown is the starting queue.
print("Workshop profiles:")
for w in WORKSHOPS.values():
    print(f"  {w.workshop_id} {w.name}: {w.status}; makes {'+'.join(sorted(w.makes))}; {w.capacity} pieces/day; "
          f"{w.lead_days}-day transport; defect rate {w.defect_rate}; cost {w.cost}/piece; "
          f"batch cap {w.max_batch}; queue {w.queue0} days")

print("\nExample order row (pieces is now a number and due_date a date):")
print(json.dumps(next(iter(ORDERS.values())), indent=2, ensure_ascii=False, default=str))
print(f"\nProcessing runs are read from: {PROCESSING_DIR}")
print(f"Plans are saved to: {OUTPUT_DIR}")

## 2. Read only a completed, matching processing handoff

- `load_handoff` follows `current-run.json` to the latest started processing run. Processing updates this pointer when a run starts.
- No started run, no summary yet, or an incomplete run: it returns no handoff and a message saying why. An incomplete run is reported with its status and verified count (B4). An older run is never used instead.
- A completed run is accepted only if every check below passes. Otherwise it raises an error:
  - the saved task state matches its fingerprint, and the CSV files match theirs, so an old verification cannot be paired with changed evidence or changed order and workshop data;
  - every task has a current verification, or (schema 4 only) a current review instead, and never both;
  - the intake, task and link review lists match the saved processing state (A9);
  - schema 4 only: every input message and every extracted item has an outcome, and tasks that unresolved input could change are held for review.
- Historical schema 3 handoffs stay readable.
- Non-task messages are saved only in the processing state, not in the handoff file. `load_handoff` copies them, with their text from the run's `manifest.json`, so the final table can account for every chat message (B5). Allocation does not reinterpret them.
- Processing task and link reviews are shown in the plan, but never enter workshop selection or reserve queue time.
- `digest` is the same JSON fingerprint that processing uses.

The run cell shows: the handoff counts, the ready task IDs, the first ready task as JSON and the non-task messages. Without a completed run, the reason is printed; Sections 3 to 7 still run on made-up tasks.

In [ ]:
# Section 2 definitions: accept the processing output only when it is complete and unchanged.
# Order of work: current-run.json -> summary.json -> order-tasks.json and state.json -> checks.


# The same JSON fingerprint that processing uses. It checks the exact saved task state.
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


# Read the latest started processing run and return (handoff, message).
# Returns (None, message) when there is no completed run to plan yet.
# Raises an error when the saved files disagree.
def load_handoff():
    # 1. No completed run yet is normal before the first processing run.
    #    B4: an incomplete run reports its status, so the reader knows what to fix.
    pointer = PROCESSING_DIR / "current-run.json"
    if not pointer.exists():
        return None, "Run processing v3 first; no v3 run has been started."
    run_id = json.loads(pointer.read_text())["run_id"]
    run_dir = PROCESSING_DIR / run_id
    print(f"Reading the latest started processing run: {run_dir}")
    summary_file = run_dir / "summary.json"
    if not summary_file.exists():
        return None, f"Processing run {run_id} has no summary yet; it is still running or stopped early."
    summary = json.loads(summary_file.read_text())
    if not summary["complete"]:
        return None, (f"Processing run {run_id} is incomplete (status {summary['status']}, "
                      f"{summary['verified_tasks']} of {summary['tasks']} tasks verified). "
                      "Fix the processing problem and rerun processing before allocation.")

    # 2. Check the handoff version, the exact saved state and the current CSV files
    #    before accepting any task.
    handoff = json.loads((run_dir / "order-tasks.json").read_text())
    state = json.loads((run_dir / "state.json").read_text())
    if handoff["schema_version"] not in (3, 4) or not handoff["complete"] or handoff["run_id"] != run_id:
        raise ValueError("The handoff does not match this completed v3 run.")
    if handoff["state_hash"] != digest(state):
        raise ValueError("Task evidence changed after publication; rerun verification.")
    for name, expected in handoff["data_hashes"].items():
        if hashlib.sha256((ROOT / "backend/data" / name).read_bytes()).hexdigest() != expected:
            raise ValueError(f"{name} changed after verification; rerun processing.")
    # Schema 3 stays readable for recorded historical runs. Schema 4 also allows a current
    # task or link review as a finished processing outcome, kept apart from verified tasks.
    if handoff["schema_version"] == 3:
        task_reviews = {}
        handoff["task_review"], handoff["link_review"] = [], []
    else:
        task_reviews = state["task_review"]
        # The review lists must match the saved state, and every task review must belong to a saved task.
        if (handoff["task_review"] != list(task_reviews.values())
                or handoff["link_review"] != list(state["unresolved_links"].values())):
            raise ValueError("The handoff's review records do not match the saved processing state.")
        if set(task_reviews) - set(state["tasks"]):
            raise ValueError("A review refers to a task outside the saved processing state.")
        # Every extracted item must be in exactly one place: a task or a link review.
        assigned = {iid for task in state["tasks"].values() for iid in task["item_ids"]}
        unresolved = set(state["unresolved_links"])
        if assigned & unresolved or assigned | unresolved != set(state["items"]):
            raise ValueError("Extracted items are not fully accounted for by tasks and link reviews.")
        # Tasks that unresolved input could change must be held for review.
        for review in [*state["intake_review"], *state["unresolved_links"].values()]:
            if any(tid not in task_reviews for tid in review["held_task_ids"]):
                raise ValueError("A task affected by unresolved input was not held for review.")
    # Each task needs a current review or a current verification, never both.
    # The handoff's tasks must be exactly the current verified records.
    verified = []
    for tid, task in state["tasks"].items():
        if tid in task_reviews:
            review = task_reviews[tid]
            if (review["task_id"] != tid or review["version"] != task["version"]
                    or task["verified"] is not None or task["verified_version"] is not None):
                raise ValueError("The handoff contains a stale or conflicting task review.")
        else:
            if task["verified"] is None or task["version"] != task["verified_version"]:
                raise ValueError("The handoff contains stale or missing task verification.")
            verified.append(task["verified"])
    if handoff["tasks"] != verified:
        raise ValueError("The handoff contains mismatched task verification.")
    # 3. A9: the intake-review messages must also be the ones saved in the fingerprinted state.
    if handoff["intake_review"] != state["intake_review"]:
        raise ValueError("The handoff's intake_review list does not match the saved processing state.")
    # 4. B5: non-task messages are saved only in the state. Copy them, with their text from the
    #    run's input record (manifest.json), so the final table can account for every chat message.
    messages = json.loads((run_dir / "manifest.json").read_text())["configuration"]["messages"]
    texts = {message["message_id"]: message["raw_text"] for message in messages}
    # Schema 4: every input message must have an outcome (an item, non-task, or intake review)
    # and must be marked processed.
    if handoff["schema_version"] == 4:
        accounted = {item["message_id"] for item in state["items"].values()}
        accounted.update(state["non_task_message_ids"])
        accounted.update(review["message_id"] for review in state["intake_review"])
        if accounted != set(texts) or set(state["processed_message_ids"]) != set(texts):
            raise ValueError("The completed handoff does not account for every input message.")
    handoff["non_task_messages"] = [{"message_id": message_id, "raw_text": texts[message_id]}
                                    for message_id in state["non_task_message_ids"]]
    handoff["input_message_count"] = len(messages)
    print(f"Handoff checks passed: {len(handoff['tasks'])} verified tasks, {len(handoff['task_review'])} task reviews, "
          f"{len(handoff['link_review'])} link reviews, {len(handoff['intake_review'])} intake-review messages "
          f"and {len(handoff['non_task_messages'])} non-task messages; "
          "matching versions, state and CSV data.")
    return handoff, f"Using completed processing run {run_id}."

In [ ]:
# Section 2 run: read and check the latest processing run.
# Section 8 calls load_handoff again, so the real plan uses the files as they are at that point.
handoff, message = load_handoff()
print(message)
if handoff is None:
    print("No handoff to plan yet. Sections 3 to 7 still run on made-up tasks; Section 8 prints this reason again.")
else:
    ready_tasks = [task for task in handoff["tasks"] if task["information_sufficient"]]
    print(f"\nProcessing run {handoff['run_id']}: schema {handoff['schema_version']}; "
          f"{handoff['input_message_count']} input messages.")
    print(f"Ready for allocation checks: {len(ready_tasks)} of {len(handoff['tasks'])} verified tasks: "
          f"{[task['task_id'] for task in ready_tasks]}")
    print("\nFirst ready task, with the fields allocation reads:")
    print(json.dumps(ready_tasks[0], indent=2, ensure_ascii=False) if ready_tasks else "  (no ready task)")
    print("\nNon-task messages copied from manifest.json:")
    for item in handoff["non_task_messages"]:
        print(f"  {item['message_id']}: {item['raw_text']}")

## 3. Apply the hard rules, then calculate the estimates

**Hard rules** (`rejection_reason`, `split_rejected`, `hard_rule_reason`)
- Checked in this order: excluded by the request, workshop status, category, batch cap, requested workshop. The first failed rule is recorded.
- A requested workshop narrows the candidates, but cannot override the other rules. If the requested workshop fails a rule, the task goes to review; the code never chooses an unapproved substitute.
- When no workshop passes, the review reason names the requested workshop and its failed rule, or lists each failed rule with the workshops it removed (A3).

**Estimates** (`candidate_metrics`), for each allowed workshop
- Delivery days = queue + production (pieces / capacity) + expected rework (production x defect rate x 0.5) + transport.
- Lateness = delivery days minus the days from the dispatch date to the due date, never below zero. On time means zero lateness.
- Cost = pieces x cost per piece, rounded to cents.
- Share = (pieces already planned at this workshop + this batch) / (all pieces planned so far + this batch).
- Only production and expected rework occupy the queue; transport does not.
- The formulas are unchanged from v2 and v3. They estimate elapsed days and do not use a working-day calendar.

The run cell shows: the hard rules for made-up tasks C1 (ORD-045) and C2 (ORD-073, requested W8), with C2's review reason; then one estimate line per workshop allowed for C1, the on-time workshops, and the first estimate as JSON.

In [ ]:
# Section 3 definitions, part 1: the hard rules decide which workshops may take a batch, before any scoring.
# split_rejected calls rejection_reason for each workshop.
# hard_rule_reason explains the result when no workshop is allowed.


# Return the first rule that stops this workshop taking the batch, or None if it passes.
# Request exclusions and workshop limits are checked before the requested workshop.
def rejection_reason(workshop, batch, excluded_ids=(), requested_workshop_id=None):
    if workshop.workshop_id in excluded_ids:
        return "excluded_by_request"
    if workshop.status != "ACTIVE":
        return f"status={workshop.status}"
    if batch["category"] not in workshop.makes:
        return f"cannot_make_category={batch['category']}"
    if workshop.max_batch is not None and batch["pieces"] > workshop.max_batch:
        return f"exceeds_batch_cap={batch['pieces']}>{workshop.max_batch}"
    # A requested workshop narrows the candidates; it cannot override the rules above.
    if requested_workshop_id is not None and workshop.workshop_id != requested_workshop_id:
        return "not_the_requested_workshop"
    return None


# Split all workshops into allowed and rejected ones, keeping each rejection's reason.
# Allowed workshops go on to the estimates; rejected ones are saved to explain the decision.
def split_rejected(workshops, batch, excluded_ids=(), requested_workshop_id=None):
    eligible, rejected = [], []
    for workshop in workshops.values():
        reason = rejection_reason(workshop, batch, excluded_ids, requested_workshop_id)
        if reason is None:
            eligible.append(workshop)
        else:
            rejected.append({"workshop_id": workshop.workshop_id, "reason": reason})
    # A6: two short lines instead of one line per workshop.
    print("  Allowed by hard rules:", [workshop.workshop_id for workshop in eligible])
    print("  Rejected:", [f"{row['workshop_id']} {row['reason']}" for row in rejected])
    return eligible, rejected


# A3: explain why no workshop passed the hard rules, so the dispatcher does not need the raw JSON.
# With a requested workshop: name it and its failed rule. The other workshops were not requested,
# so they are not listed. Without one: list each failed rule with the workshops it removed.
def hard_rule_reason(requested_workshop_id, rejected):
    reasons = {row["workshop_id"]: row["reason"] for row in rejected}
    if requested_workshop_id is not None:
        workshop = WORKSHOPS[requested_workshop_id]
        return (f"Requested {workshop.workshop_id} ({workshop.name}) cannot take this batch: "
                f"{reasons[requested_workshop_id]}.")
    workshops_by_rule = {}
    for workshop_id, reason in reasons.items():
        workshops_by_rule.setdefault(reason, []).append(workshop_id)
    details = "; ".join(f"{reason} ({', '.join(ids)})" for reason, ids in workshops_by_rule.items())
    return f"No workshop satisfies the hard rules: {details}."

In [ ]:
# Section 3 run, part 1: the hard rules for two made-up batches built from real orders.
# Each batch is what recommend (Section 5) builds for a task without a chat deadline:
# the order's category, pieces and CSV due date, sent on the dispatch date.
example_batches = {order_id: {"order_id": order_id, "category": ORDERS[order_id]["category"],
                              "pieces": ORDERS[order_id]["pieces"], "sent_date": DISPATCH_DATE,
                              "due_date": ORDERS[order_id]["due_date"]}
                   for order_id in ("ORD-045", "ORD-073")}

c1_batch = example_batches["ORD-045"]
print(f"C1: ORD-045, {c1_batch['pieces']} pieces of {c1_batch['category']}, due {c1_batch['due_date']}; "
      "no requested or excluded workshop.")
c1_allowed, c1_rejected = split_rejected(WORKSHOPS, c1_batch)

c2_batch = example_batches["ORD-073"]
print(f"\nC2: ORD-073, {c2_batch['pieces']} pieces of {c2_batch['category']}, due {c2_batch['due_date']}; "
      "requested W8.")
c2_allowed, c2_rejected = split_rejected(WORKSHOPS, c2_batch, requested_workshop_id="W8")
# recommend uses hard_rule_reason when no workshop is allowed.
if not c2_allowed:
    print("  No workshop is allowed, so the task goes to review:", hard_rule_reason("W8", c2_rejected))

In [ ]:
# Section 3 definitions, part 2: delivery days, lateness, cost and workload share for one allowed workshop.
# The formulas are unchanged from v3; only the printout is shorter (A6).
# total_pieces already includes this batch.
def candidate_metrics(workshop, batch, queue_days, cumulative_pieces, total_pieces):
    pieces = batch["pieces"]
    # Production days: pieces divided by pieces per day.
    processing = pieces / workshop.capacity
    # Expected rework days: production days x defect rate x 0.5. An expected value, not a guarantee.
    expected_rework = processing * workshop.defect_rate * 0.5
    # Delivery days = queue + production + rework + transport. A queue below zero counts as zero.
    queue = max(0.0, queue_days)
    eta = queue + processing + expected_rework + workshop.lead_days
    # Compare the unrounded delivery days with the calendar days available. Early delivery has zero lateness.
    days_until_due = (batch["due_date"] - batch["sent_date"]).days
    tardiness = max(0.0, eta - days_until_due)
    # Cost of the whole batch, rounded to cents.
    cost = round(pieces * workshop.cost, 2)
    # This workshop's share of all planned pieces if it takes this batch.
    share = (cumulative_pieces + pieces) / total_pieces
    # One line per workshop. Rounding affects only the printed numbers.
    print(f"  {workshop.workshop_id}: delivery {eta:.2f} days = queue {queue:.2f} + production {processing:.2f} "
          f"+ rework {expected_rework:.2f} + transport {workshop.lead_days}; available {days_until_due}; "
          f"late {tardiness:.2f}; cost {cost}; defect rate {workshop.defect_rate}; share {share:.2f}")

    return {
        "workshop_id": workshop.workshop_id,
        "name": workshop.name,
        # The parts of the delivery time are kept next to the total, to explain decisions later.
        "queue_days": queue,
        "processing_days": processing,
        "expected_rework_days": expected_rework,
        "lead_days": workshop.lead_days,
        "eta_days": eta,
        "days_until_due": days_until_due,
        "tardiness_days": tardiness,
        "on_time": tardiness == 0.0,
        # The defect rate is a probability (0 to 1).
        "defect_rate": workshop.defect_rate,
        "cost_per_piece": workshop.cost,
        "cost": cost,
        "share": share,
    }

In [ ]:
# Section 3 run, part 2: estimates for the workshops allowed for C1, from the starting queues.
# These are the same arguments recommend passes for a task when nothing is planned yet,
# so every share is 1.0.
print("C1 estimates for the allowed workshops:")
c1_metrics = {w.workshop_id: candidate_metrics(w, c1_batch, INITIAL_QUEUES[w.workshop_id], 0, c1_batch["pieces"])
              for w in c1_allowed}
print("On time:", [wid for wid, row in c1_metrics.items() if row["on_time"]])
print("\nFirst estimate as JSON (recommend saves these under candidate_metrics):")
print(json.dumps(next(iter(c1_metrics.values()), None), indent=2))

## 4. Rank the candidates with the existing objectives

- Each objective compares a tuple of metrics from left to right, and the smallest wins. Equal values keep the workshop file order.

| Objective | Compared in this order (smallest wins) |
|---|---|
| `minimize_tardiness` | lateness, delivery days, cost |
| `minimize_cost` | cost, delivery days, defect rate |
| `minimize_defects` | defect rate, delivery days, cost |
| `minimize_concentration` | share, delivery days, defect rate |
| `minimize_robustness` | share, lateness, defect rate |
| `hybrid` | One score: 0.5 x lateness + 0.25 x cost + 0.25 x share, each scaled to 0..1 within the candidates. It has no defect-rate weight. |

- `PREFERENCE_OBJECTIVES` maps each request's preference to its objective (B13). `PLAN_OBJECTIVE` (Section 1) replaces it for every request, and can also select the three objectives that no preference selects: `minimize_concentration`, `minimize_robustness` and `hybrid` (A7).
- `OBJECTIVE_ALIASES` accepts alternate spellings such as `minimise_cost`. They do not add objectives.
- The plan ranks only on-time workshops, so a quality or cost preference only chooses among workshops that already meet the deadline. A score never overrides a hard rule.

The run cell shows: C1's on-time workshops from Section 3, ranked under `minimize_tardiness`, `minimize_cost` and `hybrid`, with the ranking keys.

In [ ]:
# Section 4 definitions: rank the allowed workshops with one objective.
# In the plan, only on-time workshops are ranked.
# Each tuple lists the main metric, then the tie-breakers. Every metric is minimized.
OBJECTIVE_FIELDS = {
    "minimize_tardiness": ("tardiness_days", "eta_days", "cost"),
    "minimize_cost": ("cost", "eta_days", "defect_rate"),
    "minimize_defects": ("defect_rate", "eta_days", "cost"),
    "minimize_concentration": ("share", "eta_days", "defect_rate"),
    "minimize_robustness": ("share", "tardiness_days", "defect_rate"),
}
# Alternate spellings of the same objectives. They do not add new objectives.
OBJECTIVE_ALIASES = {
    "minimise_lateness": "minimize_tardiness",
    "minimise_cost": "minimize_cost",
    "minimise_defects": "minimize_defects",
    "minimise_concentration": "minimize_concentration",
    "minimise_robustness": "minimize_robustness",
}
# B13: each request's preference selects its objective, unless PLAN_OBJECTIVE replaces it.
PREFERENCE_OBJECTIVES = {"none": DEFAULT_OBJECTIVE, "fastest": "minimize_tardiness",
                         "cheapest": "minimize_cost", "quality": "minimize_defects"}
# Experimental weights for the hybrid objective: lateness, cost and workload share, after scaling.
DEFAULT_HYBRID_WEIGHTS = {"tardiness_days": 0.5, "cost": 0.25, "share": 0.25}


# Blend the scaled metric values into one score. A smaller score is better.
def hybrid_score(candidate, all_metrics, weights):
    score = 0.0
    for field, weight in weights.items():
        # Scale each metric to 0..1 within these candidates, so money, days and fractions are comparable.
        values = [metrics[field] for metrics in all_metrics.values()]
        low, high = min(values), max(values)
        # When all values are equal, the metric adds zero (and nothing is divided by zero).
        span = high - low if high > low else 1.0
        score += weight * (candidate[field] - low) / span
    return score


# Return the ranked workshop IDs and the keys used to rank them.
# hybrid gives one score per workshop; the other objectives compare metric tuples from left to right.
def rank_candidates(metrics, objective, weights):
    if objective == "hybrid":
        # Round only the blended score, so floating-point noise cannot break a real tie.
        keys = {wid: round(hybrid_score(row, metrics, weights), 12) for wid, row in metrics.items()}
    else:
        fields = OBJECTIVE_FIELDS[objective]
        keys = {wid: tuple(row[field] for field in fields) for wid, row in metrics.items()}
    # Smaller keys win; equal keys keep workshop file order. The keys are saved as evidence for the decision.
    ranking = sorted(keys, key=keys.get)
    # A6: one line. The values behind it are in the workshop lines and in ranking_keys.
    if ranking:
        definition = weights if objective == "hybrid" else OBJECTIVE_FIELDS[objective]
        print(f"  Ranking by {objective} {definition}, best first: {ranking}")
    return ranking, keys

In [ ]:
# Section 4 run: rank C1's on-time workshops from Section 3 under three objectives.
# The plan removes late workshops before ranking (Section 5), so only on-time ones are ranked here.
c1_on_time = {wid: row for wid, row in c1_metrics.items() if row["on_time"]}
print("C1 on-time workshops:", list(c1_on_time))
for example_objective in ("minimize_tardiness", "minimize_cost", "hybrid"):
    example_ranking, example_keys = rank_candidates(c1_on_time, example_objective, DEFAULT_HYBRID_WEIGHTS)
    print("    ranking_keys:", example_keys)

## 5. Recommend one task without changing the queues

- `recommend` decides one task. It never changes the queues it is given; reserving queue time is the plan's job (Section 6).
- Steps: readiness checks, order facts, hard rules, estimates, on-time filter, ranking.
- Only the checked current fields (order, deadline, requested and excluded workshops, preference) control the decision. Quotations from the chat are kept as audit evidence only.
- The deadline is the order's CSV due date, unless processing confirmed a date from the chat. `task_deadline` gives this date; Section 6 uses the same date to plan tasks earliest deadline first.
- Each decision saves the task, the resolved inputs, the rejected workshops, the estimates, the ranking keys and a reason.

| Situation | Outcome |
|---|---|
| Not ready, split requested, or an unsupported condition | Review before any calculation. The reason names what is unsupported (B9). |
| No workshop passes the hard rules | Review. The reason names the requested workshop and its failed rule, or each failed rule with its workshops (A3). |
| Every allowed workshop is late | Review. `closest_option` saves the least-late workshop, ranked like `minimize_tardiness`. A deadline on or before the dispatch date is stated as such (A1, B3). |
| The requested workshop cannot take the batch | The review also saves `option_without_request`: the best on-time workshop under the same checks, without the request (B8). Information only: the status stays review, nothing is reserved and the request is not replaced. |
| The top workshop is mentioned in the request only as a reference | Review, so the dispatcher confirms the instruction first (B17). Processing recorded the mention as neither requested nor excluded. In processing run 3, "nothing new to QuickStitch this week" (R12) was recorded this way. The reason still gives the ranking, the estimate and the late workshops, so the dispatcher also sees the other on-time options. Nothing is reserved. |
| Otherwise | Recommendation of the top-ranked on-time workshop, with up to three alternatives. The reason quotes the delivery days, days available, cost, ranking and any late workshops removed, so it can be checked against `candidate_metrics` (B7). |

- `order_snapshot` records the order's `status` and `current_stage`. A COMPLETE order adds a warning (A4).
- `review_decision` builds the review record that every decision starts from. The plan also uses it for duplicate-order reviews.

The run cell shows: made-up tasks C1 to C4 through `recommend`, each with its trace and outcome; C1's decision record as JSON; and a check that the starting queues did not change.

In [ ]:
# Section 5 definitions: make one decision record for one task, without touching the plan's queues.
# Order of work: readiness checks -> order facts -> hard rules -> estimates -> on-time filter -> ranking.
# option_without_request adds information when a requested workshop cannot take the batch.
# task_deadline gives the one deadline that recommend checks and plan_tasks sorts by (EDD).


# The deadline the plan uses for a task: the date processing confirmed from the chat, otherwise
# the order's due date in orders.csv. None when neither is known (for example a question with
# no order); such a task never reaches recommend.
def task_deadline(task):
    if task["due_date"]:
        return date.fromisoformat(task["due_date"])
    order = ORDERS.get(task["order_id"])
    return order["due_date"] if order else None


# Start a review record. It keeps the task's identity and wording, even when nothing can be calculated.
# recommend fills in more fields as it goes; plan_tasks also uses it for duplicate-order reviews.
def review_decision(task, reason):
    return {"task_id": task["task_id"], "version": task["version"], "order_id": task["order_id"],
            "task": task, "status": "review_required", "selected_workshop": None,
            "selected_metrics": None, "closest_option": None, "option_without_request": None,
            "reason": reason, "warnings": [], "candidate_metrics": {}, "ranking": [],
            "ranking_keys": {}, "rejected_candidates": [], "alternatives": []}


# B8: when the requested workshop cannot take the batch, run the same checks without the request
# (exclusions still apply) and return the best on-time option, or None if there is none.
# Information for the dispatcher only: the task stays in review, nothing is reserved,
# and the request is never silently replaced.
def option_without_request(task, batch, queues, pieces_so_far, objective):
    print("  Information only: the same checks without the requested workshop.")
    eligible, _ = split_rejected(WORKSHOPS, batch, task["excluded_workshops"])
    total_pieces = sum(pieces_so_far.values()) + batch["pieces"]
    metrics = {w.workshop_id: candidate_metrics(w, batch, queues[w.workshop_id],
                                                pieces_so_far.get(w.workshop_id, 0), total_pieces)
               for w in eligible}
    on_time = {wid: row for wid, row in metrics.items() if row["on_time"]}
    ranking, _ = rank_candidates(on_time, objective, DEFAULT_HYBRID_WEIGHTS)
    return metrics[ranking[0]] if ranking else None


# Decide one task: recommend the best allowed, on-time workshop, or send it to review with a reason.
# objective=None uses the task's own preference; plan_tasks passes an objective that is already checked.
# require_on_time=True removes late workshops before ranking. The plan always uses True.
def recommend(task, queues, pieces_so_far, objective=None, require_on_time=True):
    # 1. Readiness. A ready task can be calculated; that does not mean a workshop can take it.
    print(f"\n{task['task_id']} (version {task['version']}), order {task['order_id']}: "
          f"preference {task['preference']}; requested {task['requested_workshop']}; "
          f"excluded {task['excluded_workshops']}")
    result = review_decision(task, "")
    if task["intent"] != "allocation" or not task["information_sufficient"]:
        result["reason"] = "The request still needs clarification before allocation."
        print("  Review:", result["reason"])
        return result
    if task["wants_split"] or task["unhandled_constraints"]:
        # B9: name what is unsupported, so the dispatcher knows what to resolve.
        unsupported = (["split requested"] if task["wants_split"] else []) + task["unhandled_constraints"]
        result["reason"] = f"Not supported by this plan, so the dispatcher must decide: {'; '.join(unsupported)}."
        print("  Review:", result["reason"])
        return result

    # 2. Order facts from orders.csv. The deadline is the CSV due date unless processing
    #    confirmed a date from the chat (task_deadline, the same date plan_tasks sorts by).
    order = ORDERS[task["order_id"]]
    if order["pieces"] <= 0:
        raise ValueError("Batch pieces must be a positive integer.")
    batch = {"order_id": task["order_id"], "category": order["category"], "pieces": order["pieces"],
             "sent_date": DISPATCH_DATE, "due_date": task_deadline(task)}
    if objective is None:
        objective = PREFERENCE_OBJECTIVES[task["preference"]]
    deadline_source = "confirmed chat date" if task["due_date"] else "order CSV"
    print(f"  Order facts: {batch['pieces']} pieces of {batch['category']}, due {batch['due_date']} "
          f"({deadline_source}); status {order['status']}, stage {order['current_stage'] or '-'}")
    # A4: completed orders stay allowed in this historical experiment, but the record says so.
    if order["status"] == "COMPLETE":
        warning = "orders.csv marks this order COMPLETE; it is allowed only in this historical experiment."
        result["warnings"].append(warning)
        print("  Warning:", warning)

    # 3. Apply every hard rule before comparing time or price. Then remove late workshops.
    eligible, rejected = split_rejected(WORKSHOPS, batch, task["excluded_workshops"], task["requested_workshop"])
    metrics = {w.workshop_id: candidate_metrics(w, batch, queues[w.workshop_id],
               pieces_so_far.get(w.workshop_id, 0), sum(pieces_so_far.values()) + batch["pieces"])
               for w in eligible}
    rankable = dict(metrics)
    late = []
    if require_on_time:
        late = [wid for wid, row in metrics.items() if not row["on_time"]]
        for wid in late:
            del rankable[wid]
            rejected.append({"workshop_id": wid, "reason": "estimated_delivery_after_deadline"})
        if late:
            print("  Late, removed by the on-time rule:", late)
    ranking, keys = rank_candidates(rankable, objective, DEFAULT_HYBRID_WEIGHTS)

    # 4. Save enough numbers to explain the result later, even after the queues change.
    #    A4 adds the order's status and stage. A6 saves workshop profiles once per plan, not here.
    result.update(objective=objective, policy_version=POLICY_VERSION,
                  policy_settings={"hybrid_weights": DEFAULT_HYBRID_WEIGHTS if objective == "hybrid" else {},
                                   "pieces_so_far": dict(pieces_so_far)},
                  order_snapshot={**batch, "sent_date": batch["sent_date"].isoformat(),
                                  "due_date": batch["due_date"].isoformat(),
                                  "status": order["status"], "current_stage": order["current_stage"]},
                  queue_snapshot=dict(queues),
                  applied_constraints={"excluded_workshops": task["excluded_workshops"],
                                       "requested_workshop": task["requested_workshop"],
                                       "require_on_time": require_on_time},
                  candidate_metrics=metrics, rejected_candidates=rejected,
                  ranking=ranking, ranking_keys=keys)

    # 5. Nothing to recommend: explain why. A review reserves no queue time.
    if not ranking:
        if not eligible:
            # A3: name the failed hard rules.
            result["reason"] = hard_rule_reason(task["requested_workshop"], rejected)
        else:
            # A1: every allowed workshop is late. Name the least-late one, in minimize_tardiness order
            # (lateness, then delivery days, then cost). min() keeps file order on ties, like sorted().
            tardiness_fields = OBJECTIVE_FIELDS["minimize_tardiness"]
            closest = min(metrics.values(), key=lambda row: tuple(row[field] for field in tardiness_fields))
            result["closest_option"] = closest
            # B3: a deadline on or before the dispatch date is stated plainly, not as negative days.
            if closest["days_until_due"] <= 0:
                timing = (f"The deadline {batch['due_date']} is on or before the dispatch date {DISPATCH_DATE}, "
                          "so no workshop can deliver on time.")
            else:
                timing = (f"No workshop allowed for this request meets the deadline {batch['due_date']} "
                          f"({closest['days_until_due']} days available).")
            result["reason"] = (f"{timing} Closest option: {closest['workshop_id']} ({closest['name']}), "
                                f"delivery in {closest['eta_days']:.2f} days, "
                                f"{closest['tardiness_days']:.2f} days late.")
        # B8: the request is kept, but the dispatcher also sees the best on-time option without it.
        if task["requested_workshop"] is not None:
            option = option_without_request(task, batch, queues, pieces_so_far, objective)
            result["option_without_request"] = option
            if option is None:
                result["reason"] += " No other allowed workshop can deliver on time either."
            else:
                result["reason"] += (f" Without the request, the best on-time option is {option['workshop_id']} "
                                     f"({option['name']}), delivery in {option['eta_days']:.2f} of "
                                     f"{option['days_until_due']} available days; using it needs the "
                                     "dispatcher's approval.")
        print("  Review:", result["reason"])
        return result

    # 6. Recommend the top-ranked workshop and keep up to three alternatives for the dispatcher.
    #    B7: the reason quotes the saved numbers, so it can be checked against candidate_metrics.
    selected = ranking[0]
    chosen = metrics[selected]
    reason = (f"{selected} ({chosen['name']}) ranks first under {objective} (order: {', '.join(ranking)}): "
              f"delivery in {chosen['eta_days']:.2f} of {chosen['days_until_due']} available days, "
              f"cost {chosen['cost']}.")
    if late:
        reason += f" Removed as late: {', '.join(late)}."
    # B17: the request mentions the top workshop, but processing recorded the mention only as a
    # reference (neither requested nor excluded), so the instruction is unclear. In processing
    # run 3, "nothing new to QuickStitch this week" was recorded this way. Ask instead of
    # recommending; the reason above still shows the other on-time options. Nothing is reserved.
    referenced = {rule["workshop_id"] for rule in task["workshop_rules"] if rule["role"] == "reference"}
    if selected in referenced and selected != task["requested_workshop"]:
        result["reason"] = (f"{reason} The request mentions {selected} without a clear instruction (recorded "
                            f"as a reference, not as requested or excluded); confirm it before using {selected}.")
        print("  Review:", result["reason"])
        return result
    result.update(status="recommendation", selected_workshop=selected, selected_metrics=chosen,
                  alternatives=[metrics[wid] for wid in ranking[1:4]], reason=reason)
    print("  Recommendation:", result["reason"])
    # Reachable only with require_on_time=False, an alternative policy kept for experiments (A10).
    if chosen["tardiness_days"] > 0:
        warning = "This optional late recommendation does not meet the deadline."
        result["warnings"].append(warning)
        print("  Warning:", warning)
    return result

In [ ]:
# Section 5 run: recommend four made-up tasks, one at a time, from the starting queues.
# check_task is a test helper, not allocation code. It builds a ready allocation task with the
# handoff's fields; keyword arguments change single fields. Section 6 and Section 7 use it too.
def check_task(task_id, order_id, **changes):
    task = {"task_id": task_id, "version": 1, "order_id": order_id, "intent": "allocation",
            "information_sufficient": True, "description": f"Controlled check {task_id}",
            "source_message_ids": [task_id], "preference": "none", "requested_workshop": None,
            "excluded_workshops": [], "due_date": None, "wants_split": False,
            "unhandled_constraints": [], "doubts_to_clarify": [], "missing_details": [],
            "workshop_rules": []}
    return {**task, **changes}


example_tasks = [
    check_task("C1", "ORD-045"),                           # 150 TOPS, 8 days: W6 is fastest
    check_task("C2", "ORD-073", requested_workshop="W8"),  # 400 pieces, FreshStart's cap is 300
    check_task("C3", "ORD-083"),                           # due 23 March, before the dispatch date
    check_task("C4", "ORD-061", wants_split=True),         # splitting is not supported
]
# Nothing is planned yet: the starting queues, and no pieces placed so far.
example_decisions = {task["task_id"]: recommend(task, INITIAL_QUEUES, {}) for task in example_tasks}

print("\nOutcomes:")
for task_id, record in example_decisions.items():
    print(f"  {task_id}: {record['status']}; selected workshop: {record['selected_workshop']}")

# The saved record has more fields: task (the full task), candidate_metrics (the estimate lines
# printed above) and alternatives (the next workshops in the ranking, with their estimates).
print("\nC1's decision record as JSON, without task, candidate_metrics and alternatives:")
c1_record = {key: value for key, value in example_decisions["C1"].items()
             if key not in ("task", "candidate_metrics", "alternatives")}
print(json.dumps(c1_record, indent=2, ensure_ascii=False))
print("\nStarting queues unchanged after four decisions:",
      INITIAL_QUEUES == {wid: workshop.queue0 for wid, workshop in WORKSHOPS.items()})

## 6. Build the tentative plan

- `plan_tasks` handles the tasks **earliest deadline first (EDD)**, on its own copy of the starting queues.
  - Why: the plan is greedy. Each recommendation reserves queue time, and later tasks see that load. In chat order, a request that can wait could take the fast workshop an urgent request needs, and the urgent request would go to review as late.
  - The deadline is the one the plan checks (`task_deadline`, Section 5): a date confirmed from the chat, otherwise the CSV due date.
  - Equal deadlines keep chat order. A task with no deadline (a question or unclear request with no known order) goes last; it never reserves time.
  - Only the order changes. Routing, hard rules, estimates and ranking are the same as before. The plan prints the planning order and saves it as `planning_order`.
  - The summary table still reads in chat order.
- Routing (A2):
  - a question with no missing details or doubts is reference only (`reference_tasks`);
  - a question with missing details or doubts, and any other task that is not ready, goes to `needs_clarification`;
  - if two or more allocation tasks name the same order, the ready ones go to review, and the reason names all of them in chat order (B10). No request is silently chosen, so EDD cannot change this outcome. A history question about that order does not create this conflict;
  - every other ready task goes to `recommend` (Section 5).
- A correction already belongs to its original task, because processing linked it and verified the task again.
- Each recommendation adds its production and rework days to the plan's queue copy, so later requests see the tentative load. Transport does not occupy capacity. Reviews add nothing. Nothing here writes an accepted assignment.
- `PLAN_OBJECTIVE` is resolved and checked once, before planning: an alternate spelling is accepted, and an unknown name stops at once (A7, B13).
- `summarize_plan` builds `summary_rows`: one row per decision, clarification, reference question, intake-review message, processing task review, unresolved link and non-task message. Each row has its outcome, the workshop (or the option named in a review), delivery days, days available, its reason and every chat message it covers (A6, B5, B12).
- Processing reviews are copied into the plan separately (Section 8). They never enter `plan_tasks` and reserve no queue time.
- `code_fingerprint` identifies the code that produced a saved plan (B2). It hashes the `definitions` cells of the saved notebook file.

The run cell shows: a small plan of made-up tasks C1 and C6 to C9 (one recommendation, a duplicate-order pair, a reference question and an unclear request) with its planning order, its summary table, one summary row as JSON, and this notebook's code fingerprint.

In [ ]:
# Section 6 definitions: sort the tasks earliest deadline first, route every task, run recommend for
# the ready ones, and reserve queue time for recommendations only.
# Order of work: plan_tasks -> recommend for each task -> summarize_plan.
# code_fingerprint identifies the code that produced a saved plan.


# Build the tentative plan for all tasks, earliest deadline first (EDD).
# Returns the decisions, clarifications, reference questions, the planning order
# and the queues before and after the plan.
def plan_tasks(tasks, initial_queues, objective=None):
    # A7, B13: resolve an alternate spelling once, and stop at once on an unknown name
    # instead of failing halfway through the plan.
    if objective is not None:
        objective = OBJECTIVE_ALIASES.get(objective, objective)
        if objective not in OBJECTIVE_FIELDS and objective != "hybrid":
            raise ValueError(f"Unknown PLAN_OBJECTIVE: {objective}")
    # 1. Work on a fresh copy of the queues. Repeated previews never change the starting queues
    #    and never reserve the same time twice.
    queues = dict(initial_queues)
    pieces_so_far = {}
    decisions, clarifications, reference = [], [], []
    # B10: list the allocation tasks for each order, in chat order, so a duplicate-order review
    # can name all of them as written.
    tasks_by_order = {}
    for task in tasks:
        if task["intent"] == "allocation" and task["order_id"] is not None:
            tasks_by_order.setdefault(task["order_id"], []).append(task["task_id"])

    # EDD (earliest due date first). The plan is greedy: each recommendation reserves queue time that
    # later tasks see. In chat order, a request that can wait could take the fast workshop an urgent
    # request needs, and the urgent one would go to review as late. So tasks are sorted by the deadline
    # the plan checks (task_deadline). sorted() is stable: equal deadlines keep chat order. A task with
    # no deadline goes last; it never reserves time. Routing, rules and ranking below are unchanged.
    planning_order = sorted(tasks, key=lambda task: task_deadline(task) or date.max)

    print(f"Building a tentative plan for {len(tasks)} tasks. Objective setting: {objective}")
    print("Planning order, earliest deadline first: " + ", ".join(
        f"{task['task_id']} ({task_deadline(task) or 'no deadline'})" for task in planning_order))
    print("Starting queues:", queues)
    # 2. Route questions and tasks with missing facts before any workshop selection.
    for task in planning_order:
        open_questions = task["doubts_to_clarify"] or task["missing_details"]
        # A2: only a question with nothing left to clarify is reference material.
        if task["intent"] == "question" and not open_questions:
            reference.append(task)
            print(f"\n{task['task_id']}: reference question; excluded from allocation: {task['description']}")
            continue
        if task["intent"] == "question" or not task["information_sufficient"]:
            clarifications.append(task)
            print(f"\n{task['task_id']}: needs clarification; queues unchanged. Open questions: {open_questions}")
            continue
        # A ready task is an allocation with a known order, so tasks_by_order lists it.
        same_order = tasks_by_order[task["order_id"]]
        if len(same_order) > 1:
            decision = review_decision(task, f"Allocation requests {', '.join(same_order)} all name "
                                             f"{task['order_id']}; clarify the intended batches.")
            print(f"\n{task['task_id']}: review: {decision['reason']}")
        else:
            decision = recommend(task, queues, pieces_so_far, objective)
        decisions.append(decision)

        # 3. Only a recommendation reserves time in this tentative plan.
        #    Transport uses no production capacity, so only production and rework are added.
        if decision["status"] == "recommendation":
            wid = decision["selected_workshop"]
            chosen = decision["selected_metrics"]
            added = chosen["processing_days"] + chosen["expected_rework_days"]
            print(f"  Tentative reservation at {wid}: {queues[wid]:.3f} -> {queues[wid] + added:.3f} queue days "
                  f"(+{added:.3f} production/rework days).")
            queues[wid] += added
            pieces_so_far[wid] = pieces_so_far.get(wid, 0) + decision["order_snapshot"]["pieces"]
        else:
            print("  Review item: no queue time reserved.")
    print("\nPlan decision counts:", dict(Counter(record["status"] for record in decisions)))
    print(f"Needs clarification: {len(clarifications)}; reference questions: {len(reference)}")
    print("Final tentative queues:", queues)
    # A6: workshop profiles are saved once here instead of inside every decision.
    # planning_order records the EDD order, so a reader can see which tasks reserved time first.
    return {"dispatch_date": DISPATCH_DATE.isoformat(), "policy_version": POLICY_VERSION,
            "objective_setting": objective,
            "planning_order": [task["task_id"] for task in planning_order],
            "workshop_profiles": {wid: {**asdict(w), "makes": sorted(w.makes)} for wid, w in WORKSHOPS.items()},
            "decisions": decisions, "needs_clarification": clarifications, "reference_tasks": reference,
            "queues_before_plan": dict(initial_queues), "queues_after_plan": queues}


# A6: one row per task or message, saved with the plan and printed as the final table.
# "id" is a task ID, an item ID (link review), or a message ID (intake review or non-task message).
# "messages" lists every chat message the row covers (B5, B12).
# For a review, the workshop columns show the option named in its reason: the best option
# without the request (B8), otherwise the closest late option (A1).
# Empty fields are None in the file and "-" in the table.
def summarize_plan(plan):
    rows = []
    for record in plan["decisions"]:
        option = record["selected_metrics"] or record["option_without_request"] or record["closest_option"] or {}
        rows.append({"id": record["task_id"], "order_id": record["order_id"], "outcome": record["status"],
                     "workshop": option.get("workshop_id"), "delivery_days": option.get("eta_days"),
                     "days_available": option.get("days_until_due"), "reason": record["reason"],
                     "messages": record["task"]["source_message_ids"]})
    for task in plan["needs_clarification"]:
        # B6: an unclear request can arrive with no recorded question; the row still says what to do.
        questions = "; ".join(task["doubts_to_clarify"] or task["missing_details"])
        fallback = f"Intent {task['intent']}; no question was recorded, so ask the requester."
        rows.append({"id": task["task_id"], "order_id": task["order_id"], "outcome": "needs_clarification",
                     "workshop": None, "delivery_days": None, "days_available": None,
                     "reason": questions or fallback, "messages": task["source_message_ids"]})
    for task in plan["reference_tasks"]:
        # B11: the question is recorded for the dispatcher, not answered.
        rows.append({"id": task["task_id"], "order_id": task["order_id"], "outcome": "reference",
                     "workshop": None, "delivery_days": None, "days_available": None,
                     "reason": f"Information question; not an allocation, and not answered here: {task['description']}",
                     "messages": task["source_message_ids"]})
    for review in plan["intake_review"]:
        rows.append({"id": review["message_id"], "order_id": None, "outcome": "intake_review",
                     "workshop": None, "delivery_days": None, "days_available": None,
                     "reason": f"Processing could not read this message reliably: {review['raw_text']}",
                     "messages": [review["message_id"]]})
    # Processing task and link reviews never enter plan_tasks: their interpretations are not
    # trusted enough for allocation, so they have no selected workshop or queue reservation.
    for review in plan.get("task_review", []):
        rows.append({"id": review["task_id"], "order_id": None, "outcome": "processing_review",
                     "workshop": None, "delivery_days": None, "days_available": None,
                     "reason": review["reason"], "messages": review["source_message_ids"]})
    for review in plan.get("link_review", []):
        rows.append({"id": review["item_id"], "order_id": None, "outcome": "link_review",
                     "workshop": None, "delivery_days": None, "days_available": None,
                     "reason": review["reason"], "messages": [review["message_id"]]})
    for message in plan["non_task_messages"]:
        # B5: messages in which processing found no request or question.
        rows.append({"id": message["message_id"], "order_id": None, "outcome": "non_task",
                     "workshop": None, "delivery_days": None, "days_available": None,
                     "reason": f"Processing found no request or question: {message['raw_text']}",
                     "messages": [message["message_id"]]})
    # Sort by chat line (L0001, L0002, ...) so the table reads in message order.
    rows.sort(key=lambda row: row["id"].removeprefix("T:"))

    print(f"\n{'ID':<11} {'Order':<8} {'Outcome':<19} {'Workshop':<8} {'Delivery':>8} {'Available':>9}  Messages")
    for row in rows:
        delivery = "-" if row["delivery_days"] is None else f"{row['delivery_days']:.2f}"
        available = "-" if row["days_available"] is None else row["days_available"]
        print(f"{row['id']:<11} {row['order_id'] or '-':<8} {row['outcome']:<19} "
              f"{row['workshop'] or '-':<8} {delivery:>8} {available:>9}  {', '.join(row['messages'])}")
        print(f"    {row['reason']}")
    return rows


# B2: a fingerprint of this notebook's definitions cells, saved with each plan to identify the code
# that produced it (the same method processing uses for its code_hash). It reads the saved .ipynb
# file, so unsaved editor changes are not included. Any edit to a definitions cell, even a comment,
# changes the value.
def code_fingerprint():
    notebook = json.loads(NOTEBOOK_FILE.read_text())
    return digest([cell["source"] for cell in notebook["cells"]
                   if "definitions" in cell.get("metadata", {}).get("tags", [])])

In [ ]:
# Section 6 run: a small plan of made-up tasks, to show the planning order (earliest deadline first),
# routing, reservation and the summary table.
example_plan = plan_tasks([
    check_task("C1", "ORD-045"),                                                   # one recommendation
    check_task("C6", "ORD-099"), check_task("C7", "ORD-099"),                      # two requests for one order
    check_task("C8", "ORD-020", intent="question", information_sufficient=False),  # nothing to ask
    check_task("C9", None, intent="unclear", information_sufficient=False),        # nothing recorded
], INITIAL_QUEUES)
# summarize_plan also reads the processing lists, which plan_tasks does not create.
# Section 8 copies them from the handoff; this example has none.
example_plan["intake_review"], example_plan["non_task_messages"] = [], []
example_rows = summarize_plan(example_plan)
print("\nFirst summary row as JSON (Section 8 saves these as summary_rows):")
print(json.dumps(example_rows[0], indent=2, ensure_ascii=False))

if NOTEBOOK_FILE.exists():
    print(f"\ncode_fingerprint() of the saved {NOTEBOOK_FILE.name}: {code_fingerprint()}")
else:
    print(f"\n{NOTEBOOK_FILE} was not found. Save this notebook there before Section 8, which saves the fingerprint.")

## 7. Check the rules on made-up tasks

- These checks run before the real plan, so a broken rule stops the notebook before anything is saved (B16).
- Each made-up task uses a real order and the real workshop data. Its expected answer was worked out by hand from the CSV values.

| Task | Order | Expected result |
|---|---|---|
| C1 | ORD-045, 150 TOPS, 8 days available | Recommended to W6; only production and rework are added to W6's queue copy |
| C2 | ORD-073, requested W8 | Review: 400 pieces exceed FreshStart's 300 cap; an on-time option without the request is named |
| C3 | ORD-083, due 23 March | Review: the deadline is before the dispatch date; closest option W1 |
| C4 | ORD-061, split requested | Review before any calculation |
| C5 | ORD-103, requested W2 | Review: W2 is allowed but late, and no other workshop is on time either |
| C6, C7 | ORD-099 twice | Both reviewed; the reason names both requests |
| C8 | A question with nothing to ask | Reference only |
| C9 | An unclear request with nothing recorded | Needs clarification, with a fallback reason |
| C10 | ORD-053, QuickStitch mentioned only as a reference | Review: QuickStitch ranks first but its instruction is unclear; nothing is reserved |
| C11 to C14 | Confirmed chat deadlines 7, 5 and 5 April, then no deadline | EDD planning order C12, C13, C11, C14: earliest first, ties in chat order, no deadline last, and the chat date replaces the CSV date |

- C1 is planned on its own first. The answers for C2 to C10 were worked out with C1's W6 reservation already in the queues, as in v4's chat order. EDD would plan C3 (due 23 March) before C1, so C2 to C10 start from the queues after C1. None of them reserves time, so their own order cannot change an answer.

- They also check an objective alias, an unknown objective, a tie-break, and that the starting queues never change.
- The printed trace is captured instead of shown. Run `print(trace.getvalue())` to read it.
- `check_task` is defined in Section 5.
- These checks test the notebook's rules only. They are not an allocation run, and say nothing about how well processing reads the chat.

The run cell shows: the expected error for an unknown objective, then "All controlled checks passed". A failed check stops here with an `AssertionError`.

In [ ]:
# Section 7 run: made-up tasks with known answers, using the real CSV values (B16).
# A failed check stops the notebook here, before anything is saved.
# These checks test this notebook's rules only; they are not an allocation run.
import contextlib
import io

# Two workshops with identical ranking metrics, for the tie-break check.
tied = {"W2": {"tardiness_days": 0.0, "eta_days": 3.0, "cost": 100.0},
        "W1": {"tardiness_days": 0.0, "eta_days": 3.0, "cost": 100.0}}
# Run the made-up plans with their printed trace captured; print(trace.getvalue()) shows it.
with contextlib.redirect_stdout(io.StringIO()) as trace:
    # C1 is planned first, on its own. The answers for C2 to C10 were worked out with C1's W6
    # reservation already in the queues (in v4 they came after C1 in the chat). EDD plans C3
    # (due 23 March), and maybe others, before C1, so C2 to C10 start from the queues after C1.
    # None of them reserves time, so their own planning order cannot change any answer.
    first = plan_tasks([check_task("C1", "ORD-045")], INITIAL_QUEUES)  # 150 TOPS, 8 days: W6 is fastest
    checks = plan_tasks([
        check_task("C2", "ORD-073", requested_workshop="W8"),  # 400 pieces, FreshStart's cap is 300
        check_task("C3", "ORD-083"),                           # due 23 March, before the dispatch date
        check_task("C4", "ORD-061", wants_split=True),         # splitting is not supported
        check_task("C5", "ORD-103", requested_workshop="W2"),  # W2 allowed but late; nobody is on time
        check_task("C6", "ORD-099"), check_task("C7", "ORD-099"),  # two requests for one order
        check_task("C8", "ORD-020", intent="question", information_sufficient=False),  # nothing to ask
        check_task("C9", None, intent="unclear", information_sufficient=False),        # nothing recorded
        check_task("C10", "ORD-053", workshop_rules=[{"workshop_id": "W1", "evidence_id": "C10:E1",
                                                      "role": "reference", "superseded_by": None}]),
    ], first["queues_after_plan"])
    checks["intake_review"], checks["non_task_messages"] = [], []
    rows = {row["id"]: row for row in summarize_plan(checks)}
    # EDD: confirmed chat dates set these deadlines, so the expected order needs no CSV values.
    edd_plan = plan_tasks([
        check_task("C11", "ORD-053", due_date="2026-04-07"),  # first in the chat, but due later
        check_task("C12", "ORD-073", due_date="2026-04-05"),  # earliest deadline
        check_task("C13", "ORD-045", due_date="2026-04-05"),  # same deadline as C12; CSV date is 9 April
        check_task("C14", None, intent="unclear", information_sufficient=False),  # no deadline
    ], INITIAL_QUEUES)
    alias_plan = plan_tasks([], INITIAL_QUEUES, "minimise_cost")
    tie_ranking, _ = rank_candidates(tied, "minimize_tardiness", DEFAULT_HYBRID_WEIGHTS)
decision = {record["task_id"]: record for record in first["decisions"] + checks["decisions"]}
queues_after_c1 = first["queues_after_plan"]

# C1: recommended to W6; only production and expected rework are added, never transport.
assert decision["C1"]["selected_workshop"] == "W6"
w6 = decision["C1"]["selected_metrics"]
assert queues_after_c1["W6"] == INITIAL_QUEUES["W6"] + (w6["processing_days"] + w6["expected_rework_days"])
assert all(queues_after_c1[wid] == INITIAL_QUEUES[wid] for wid in INITIAL_QUEUES if wid != "W6")
# C2 to C10 are reviews, clarifications or reference questions: they reserve nothing.
# The starting queues themselves never change.
assert checks["queues_after_plan"] == queues_after_c1
assert INITIAL_QUEUES == {wid: workshop.queue0 for wid, workshop in WORKSHOPS.items()}
# C2 and C5: a requested workshop is never replaced; the option without it is information only.
assert decision["C2"]["status"] == "review_required"
assert "exceeds_batch_cap=400>300" in decision["C2"]["reason"]
assert decision["C2"]["option_without_request"]["on_time"]
assert decision["C5"]["closest_option"]["workshop_id"] == "W2"
assert decision["C5"]["option_without_request"] is None
# C3: a past deadline is a review that names the least-late option.
assert decision["C3"]["closest_option"]["workshop_id"] == "W1"
assert "on or before the dispatch date" in decision["C3"]["reason"]
# C4, C6 and C7: unsupported requests and duplicate orders go to review before any calculation.
assert "split requested" in decision["C4"]["reason"] and decision["C4"]["candidate_metrics"] == {}
assert all("C6, C7" in decision[task_id]["reason"] for task_id in ("C6", "C7"))
# C8 and C9: a question with nothing to ask is reference; an unclear request still gets a reason.
assert [task["task_id"] for task in checks["reference_tasks"]] == ["C8"]
assert rows["C9"]["outcome"] == "needs_clarification" and rows["C9"]["reason"]
# C10: a top workshop mentioned only as a reference is not recommended, so W1 is not reserved (checked above).
assert decision["C10"]["status"] == "review_required"
assert "recorded as a reference" in decision["C10"]["reason"]
# C11 to C14, EDD: the earliest deadline is planned first, equal deadlines keep chat order (C12 before
# C13), and a task with no deadline goes last. C13 before C11 shows that the confirmed chat date
# (5 April) is used, not the CSV due date (9 April), which would put C13 after C11 (7 April).
assert edd_plan["planning_order"] == ["C12", "C13", "C11", "C14"]
# Objectives: an alternate spelling is resolved; an unknown name stops before planning.
assert alias_plan["objective_setting"] == "minimize_cost"
try:
    plan_tasks([], INITIAL_QUEUES, "fastest")
    raise AssertionError("An unknown objective was accepted.")
except ValueError as error:
    print("Expected error for an unknown objective:", error)
# Equal ranking keys keep workshop file order.
assert tie_ranking == ["W2", "W1"]
print("All controlled checks passed. These were made-up tasks; the real plan follows.")

## 8. Read the completed run and save its plan

- The run cell calls `load_handoff` again, so the plan always uses the processing files as they are now, never an old task list in memory.
- If processing has not finished, the reason is printed and nothing is saved.
- A completed plan is saved as `output/order-allocation-v5/<processing run id>.json`. A plan made with `PLAN_OBJECTIVE` adds the objective to the file name, so the default plan is never overwritten (B1). Earlier plans are not touched.
- The plan is written to a temporary file first and then renamed, so a failed write never leaves half a plan.
- The saved plan also records `processing_run_id`, `handoff_state_hash`, `data_hashes` and `code_hash` (B2). `code_hash` covers the `definitions` cells in the saved notebook file, so save the notebook before running it after an edit.
- Processing reviews (`intake_review`, `task_review`, `link_review`) and non-task messages (`non_task_messages`) are copied with their original evidence and reasons, for the dispatcher. They never enter `plan_tasks`, so they reserve no queue time. The final table lists them with the tasks and reports how many chat messages it covers.
- To understand one decision, read `task`, `applied_constraints`, `candidate_metrics`, `ranking_keys`, `closest_option`, `option_without_request` and `reason` together. `planning_order` and the decision's `queue_snapshot` show which earlier-deadline tasks had already reserved time.

The run cell shows: the plan trace for every task, the summary table, the message coverage and the saved file path. The second run cell reads the saved file back and prints its main fields.

In [ ]:
# Section 8 run: load the current processing run every time; never reuse an old task list in memory.
handoff, message = load_handoff()
print(message)
plan = None
if handoff is not None:
    plan = plan_tasks(handoff["tasks"], INITIAL_QUEUES, PLAN_OBJECTIVE)
    # B2: record which processing run, input fingerprints and code produced this plan.
    plan.update(processing_run_id=handoff["run_id"], handoff_state_hash=handoff["state_hash"],
                data_hashes=handoff["data_hashes"], code_hash=code_fingerprint(),
                input_message_count=handoff["input_message_count"])
    # Processing reviews and non-task messages pass to the dispatcher as written.
    # They never enter plan_tasks, so they reserve no queue time.
    plan["intake_review"] = handoff["intake_review"]
    plan["task_review"] = handoff["task_review"]
    plan["link_review"] = handoff["link_review"]
    plan["non_task_messages"] = handoff["non_task_messages"]
    print(f"Processing reviews kept outside allocation: {len(plan['task_review'])} tasks and {len(plan['link_review'])} follow-ups.")
    # A6: one row per task or message, printed as a table and saved with the plan.
    plan["summary_rows"] = summarize_plan(plan)
    covered = {message_id for row in plan["summary_rows"] for message_id in row["messages"]}
    print(f"\nChat messages covered by the table: {len(covered)} of {plan['input_message_count']}")
    # B1: a plan made with PLAN_OBJECTIVE gets its own file, so the default plan is never replaced.
    if plan["objective_setting"] is None:
        file_name = f"{handoff['run_id']}.json"
    else:
        file_name = f"{handoff['run_id']}-{plan['objective_setting']}.json"
    # Write a temporary file first, then replace, so a failed write never leaves half a plan.
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    destination = OUTPUT_DIR / file_name
    temporary = destination.with_suffix(".tmp")
    temporary.write_text(json.dumps(plan, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(destination)
    print(f"\nSaved plan: {destination}")

In [ ]:
# Section 8 run, part 2: read the saved plan file back and show its main fields.
if plan is None:
    print("No plan was saved: see the message printed by the run cell above.")
else:
    saved_plan = json.loads(destination.read_text(encoding="utf-8"))
    print(f"Saved file: {destination.name}")
    for key in ("processing_run_id", "objective_setting", "policy_version", "dispatch_date",
                "input_message_count", "handoff_state_hash", "code_hash"):
        print(f"  {key}: {saved_plan[key]}")
    print("  data_hashes:", saved_plan["data_hashes"])
    print("  planning_order (earliest deadline first):", saved_plan["planning_order"])
    print("  Decisions by status:", dict(Counter(record["status"] for record in saved_plan["decisions"])))
    print("  Entries in the other lists:", {key: len(saved_plan[key]) for key in (
        "needs_clarification", "reference_tasks", "intake_review", "task_review", "link_review",
        "non_task_messages", "summary_rows")})
    print("  queues_before_plan:", saved_plan["queues_before_plan"])
    print("  queues_after_plan: ", saved_plan["queues_after_plan"])
    print("  All top-level fields:", list(saved_plan))

## 9. Inspect one task or message

- Set `LOOKUP_ID` to a task ID printed above, a link-review item ID, or a message ID from link or intake review or the non-task list (for example `L0002`).
- The cell searches every list in the plan, so the ID is found wherever it was routed (A8).
- The outcome and reason print first, then the full saved record (B15). Clarification and reference entries keep the full processing record.

The run cell shows: the outcome and reason of `LOOKUP_ID`, then its full saved record as JSON (an empty list if the ID is not in the plan).

In [ ]:
# Section 9 run: find one task or message in the saved plan.
# A8: search every list in the plan, so a task or message is found wherever it was routed.
# B15: the outcome and reason print first, then the full saved record as evidence.
LOOKUP_ID = "T:L0021:J1"
if plan is None:
    print("No plan in memory: see the message printed by the run cell above.")
else:
    for row in plan["summary_rows"]:
        if row["id"] == LOOKUP_ID:
            print(f"{row['outcome']}: {row['reason']}\n")
    tasks_and_decisions = plan["decisions"] + plan["needs_clarification"] + plan["reference_tasks"] + plan["task_review"]
    matches = [record for record in tasks_and_decisions if record["task_id"] == LOOKUP_ID]
    matches += [item for item in plan["intake_review"] + plan["non_task_messages"]
                if item["message_id"] == LOOKUP_ID]
    matches += [item for item in plan["link_review"] if LOOKUP_ID in (item["item_id"], item["message_id"])]
    print(json.dumps(matches, indent=2, ensure_ascii=False))

## Limits and reasoning to revisit

- The parser can still misunderstand a sentence; source and schema checks do not prove meaning (P2 to P4 in the issues file). In processing run 3 it recorded "nothing new to QuickStitch this week" (R12) as a reference instead of an exclusion. B17 sends that case to review, but other misreadings remain possible.
- Allocation plans only a complete processing run. Recorded interpretation reviews count as completed outcomes and reserve nothing. If the latest run is incomplete, this notebook prints the reason and saves nothing (run 2 stopped this way on the P5 deadline check; P5 is fixed and run 3 completed).
- Deadlines are elapsed-time estimates from one dispatch date, not delivery guarantees or working-day calculations.
- The on-time test (delivery days at most the days available, unrounded) is stricter than the simulator's late flag, which rounds the finish day. The simulator also replays each order from its own order date. Compare them before using these functions as a harness allocator.
- The closest option and the option without the request are information, not recommendations. Accepting late work or another workshop is a dispatcher decision.
- Completed CSV orders are allowed here as historical experiments, not approved for live redispatch.
- Standing instructions such as "nothing new to QuickStitch this week" are not applied across requests (A5, needs a decision). B17 only affects the request that mentions the workshop; another request can still be recommended to QuickStitch or list it as an alternative.
- Splitting, partial batches, unrepresented constraints and unresolved corrections stay in review.
- This historical notebook has a sequential, tentative plan without approval or reallocation. The current application implements dispatch-table approval/reallocation; live operational reservations remain future work.
- EDD is a greedy rule, not an optimum. It protects urgent requests from requests that can wait, but it can lose too: if the earlier-deadline request takes the only workshop that a slightly later, larger request could use on time, and had another on-time option itself, chat order would have placed both. Fixing that needs a look-ahead or an assignment optimiser; evaluate before adding one.
- EDD reorders only the tasks of one chat file in this preview. The 120-order simulator replays orders in `order_date` order and asks for one workshop per batch, so EDD does not change harness results. A live system that receives messages one at a time could reorder only the requests waiting together.
- Benchmark the preserved objectives separately with the unchanged 120-order simulator and all six official metrics. This notebook is not a benchmark result.

